# 04 - Portfolio baseline (long-only top-k)

Step 4 of `THESIS_GUIDE.md`. The question: **if we invested using each model's predictions, how would it go?**

**The rule** (FinBench's long-only evaluation): every 5 business days (the horizon L), take the model's latest prediction, buy the **k stocks with the highest score** in equal parts (k = 5 for DJI, 10 for NASDAQ-100 and EURO STOXX 50), hold them until the next rebalance, and repeat over the test years. A stock earns its open-to-close return on its first day (the portfolio buys at the open), then close-to-close returns.

**Benchmarks**, on the same days: *equal weight*, all the stocks the model ranks in equal parts (what you would earn with no skill at choosing among them), and the *index* itself.

**Why it matters for the thesis**: it turns predictions into allocation decisions. The *margin* of each held stock (how far its score could fall before another stock overtakes it) and the *overlap* between portfolios (of different seeds or models) measure how fragile those decisions are; later steps explain them (Step 7) and perturb them (Step 9).

Code: `xaifin/portfolio/topk.py` (holdings, margins, overlap) and `backtest.py` (returns, metrics, benchmarks); `scripts/run_portfolio.py` runs every trained model; results in `results/portfolio/`.

In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from xaifin import viz
from xaifin.config import CORE_UNIVERSES, RESULTS_ROOT, TOP_K, UNIVERSE_NAMES
from xaifin.portfolio.backtest import asset_returns, find_portfolios, performance, portfolio_returns
from xaifin.portfolio.topk import load_predictions, overlap, rebalance_periods, topk_holdings
from finbench_reference import finbench_portfolio

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
portfolios = find_portfolios()
portfolios = portfolios[(portfolios["seq_len"] == 20) & (portfolios["pred_len"] == 5)]
print(f"{len(portfolios)} portfolios:", sorted(set(zip(portfolios["model"], portfolios["universe"]))))

3 portfolios: [('MASTER', 'dji'), ('MASTER', 'nasdaq100'), ('MASTER', 'sx5e')]


## 1. Same results as FinBench's evaluation code

On one model (MASTER, DJI, seed 42, test years 2020-2024), the holdings and daily returns are computed both with FinBench's own code (`Evaluation/portfolio/transforms.py`, `returns.py`, loaded read-only through `finbench_reference.py`) and with `xaifin`, and the metrics both with FinBench's copy of quantstats and with `backtest.performance`.

In [2]:
runs = [RESULTS_ROOT / f"Regression/MASTER/dji/sl20_pl5/seed42/y{y}" for y in range(2020, 2025)]
fb_history, fb_returns, qs = finbench_portfolio([next(r.glob("results_*.pkl")) for r in runs], top_k=5,
                                                 start_date="2020-01-01", end_date="2024-12-31", pred_len=5, universe="dji")
holdings = topk_holdings(load_predictions(runs), rebalance_periods("2020-01-01", "2024-12-31", 5), k=5)
ours_sets = holdings[holdings["held"]].groupby("start")["ticker"].agg(set)
ours_returns = portfolio_returns(holdings, *asset_returns("dji"))
ours, theirs = performance(ours_returns), {"CAGR": qs.stats.cagr(fb_returns), "Sharpe": qs.stats.sharpe(fb_returns),
                                           "Sortino": qs.stats.sortino(fb_returns), "max_drawdown": qs.stats.max_drawdown(fb_returns),
                                           "volatility": qs.stats.volatility(fb_returns)}
print("rebalances:", len(fb_history), "| identical holdings:", all(set(s["tickers"]) == ours_sets[s["test_start"]] for s in fb_history))
print("days:", len(fb_returns), "| same dates:", fb_returns.index.equals(ours_returns.index),
      "| largest daily-return difference:", float((fb_returns - ours_returns).abs().max()))
pd.DataFrame({"xaifin": ours, "FinBench + quantstats": theirs}).loc[list(theirs)]

rebalances: 260 | identical holdings: True
days: 1254 | same dates: True | largest daily-return difference: 2.7755575615628914e-17


,xaifin,FinBench + quantstats
CAGR,0.0784,0.0784
Sharpe,0.4341,0.4341
Sortino,0.6550,0.6550
max_drawdown,-0.2945,-0.2945
volatility,0.2394,0.2394


## 2. Performance per universe

Per model: compound annual growth (CAGR), Sharpe and Sortino ratios (risk-adjusted return, rf = 0), the worst fall from a peak (max drawdown), annualized volatility, and turnover (the average share of the portfolio replaced at each weekly rebalance). The last rows are the benchmarks.

In [3]:
def load(path):
    import json
    return (json.loads((path / "metrics.json").read_text()), pd.read_parquet(path / "returns.parquet"),
            pd.read_parquet(path / "holdings.parquet"))

loaded = {(r.model, r.universe, r.seed): load(r.path) for r in portfolios.itertuples()}
summary_rows = []
for universe in CORE_UNIVERSES:
    keys = [key for key in loaded if key[1] == universe and key[2] == 42]
    if not keys:
        continue
    keys.sort(key=lambda key: viz.MODEL_ORDER.index(key[0]))
    first = loaded[keys[0]]
    rows = {key[0]: loaded[key][0]["portfolio"] for key in keys}
    rows["equal weight"], rows["index"] = first[0]["equal_weight"], first[0]["index"]
    table = pd.DataFrame(rows).T[["CAGR", "Sharpe", "Sortino", "max_drawdown", "volatility", "turnover"]]
    summary_rows += [{"universe": universe, "portfolio": name, **values} for name, values in table.to_dict("index").items()]
    years = first[0]["setup"]["test_years"]
    print(f"{UNIVERSE_NAMES[universe]}, top {TOP_K[universe]}, {years[0]}-{years[-1]}")
    display(table.style.format({"CAGR": "{:+.2%}", "Sharpe": "{:.2f}", "Sortino": "{:.2f}", "max_drawdown": "{:.1%}",
                                "volatility": "{:.1%}", "turnover": "{:.0%}"}, na_rep="–"))
    curves = pd.DataFrame({key[0]: (1 + loaded[key][1].set_index("date")["portfolio"]).cumprod() for key in keys})
    benchmarks = first[1].set_index("date")
    curves["equal weight"], curves["index"] = (1 + benchmarks["equal_weight"]).cumprod(), (1 + benchmarks["index"]).cumprod()
    viz.equity_chart(curves, f"{UNIVERSE_NAMES[universe]}: top-{TOP_K[universe]} portfolios").show()

DJI, top 5, 2020-2024


,CAGR,Sharpe,Sortino,max_drawdown,volatility,turnover
MASTER,+7.84%,0.43,0.65,-29.4%,23.9%,66%
equal weight,+5.89%,0.39,0.56,-31.2%,19.3%,–
index,+8.73%,0.51,0.71,-37.1%,20.8%,–


NASDAQ-100, top 10, 2020-2024


,CAGR,Sharpe,Sortino,max_drawdown,volatility,turnover
MASTER,+9.02%,0.43,0.62,-48.4%,32.3%,72%
equal weight,+9.62%,0.51,0.72,-37.6%,23.6%,–
index,+20.16%,0.84,1.18,-35.6%,26.1%,–


EURO STOXX 50, top 10, 2021-2024


,CAGR,Sharpe,Sortino,max_drawdown,volatility,turnover
MASTER,+11.98%,0.68,1.00,-26.9%,19.0%,61%
equal weight,+10.89%,0.72,1.04,-23.2%,15.7%,–
index,+8.20%,0.54,0.77,-25.5%,16.9%,–


In [4]:
yearly = pd.DataFrame({f"{key[0]} ({key[1]})": data[0]["yearly_returns"]["portfolio"] for key, data in loaded.items() if key[2] == 42})
yearly.T.style.format("{:+.1%}", na_rep="–")

,2020,2021,2022,2023,2024
MASTER (dji),+19.6%,-2.2%,+0.6%,+18.1%,+4.7%
MASTER (nasdaq100),+26.9%,+14.8%,-25.3%,+41.8%,-0.3%
MASTER (sx5e),–,+29.2%,-7.1%,+21.2%,+7.8%


## 3. How fragile are the choices? Margins to the k-boundary

For every held stock and week: how much its score could fall before another stock overtakes it, in standard deviations of that day's scores (`margin_z`). A small margin means a small change in the prediction (another seed, a little noise in the data) would change the portfolio. The table gives the median margin and the share of holdings within 0.1 standard deviations of dropping out.

In [5]:
margin_rows = []
for (model, universe, seed), (_, _, h) in loaded.items():
    m = h.loc[h["held"], "margin_z"]
    margin_rows.append({"model": model, "universe": universe, "seed": seed, "holdings": len(m), "median margin": m.median(),
                        "share below 0.1": (m < 0.1).mean(), "share below 0.25": (m < 0.25).mean()})
margins = pd.DataFrame(margin_rows)
display(margins.style.format({"median margin": "{:.3f}", "share below 0.1": "{:.0%}", "share below 0.25": "{:.0%}"}))
model, universe, seed = next(iter(loaded))
viz.margin_histogram(loaded[(model, universe, seed)][2], f"{model}, {UNIVERSE_NAMES[universe]}: margins of the held stocks").show()

,model,universe,seed,holdings,median margin,share below 0.1,share below 0.25
0,MASTER,dji,42,1300,0.417,15%,33%
1,MASTER,nasdaq100,42,2600,0.418,15%,33%
2,MASTER,sx5e,42,2080,0.460,13%,30%


## 4. Do different models pick the same stocks?

For each pair of models in a universe: the average overlap (Jaccard: shared stocks / stocks held by either) of their portfolios over the common weeks. Two portfolios of k stocks drawn at random from N would overlap by about k / (2N - k) on average (about 0.09 for DJI, 0.05 for NASDAQ-100 and 0.11 for EURO STOXX 50), the no-agreement reference.

In [6]:
for universe in CORE_UNIVERSES:
    keys = sorted([key for key in loaded if key[1] == universe and key[2] == 42], key=lambda key: viz.MODEL_ORDER.index(key[0]))
    if len(keys) < 2:
        print(f"{UNIVERSE_NAMES[universe]}: fewer than two models with a complete portfolio")
        continue
    names = [key[0] for key in keys]
    matrix = pd.DataFrame(np.nan, index=names, columns=names)
    for a in keys:
        for b in keys:
            matrix.loc[a[0], b[0]] = overlap(loaded[a][2], loaded[b][2])["jaccard"].mean()
    viz.overlap_heatmap(matrix, f"{UNIVERSE_NAMES[universe]}: average overlap of the stocks held, by pair of models").show()
    display(matrix)

DJI: fewer than two models with a complete portfolio
NASDAQ-100: fewer than two models with a complete portfolio
EURO STOXX 50: fewer than two models with a complete portfolio


## 5. Does the same model pick the same stocks with another seed?

The same comparison between the seeds of one model (0, 5, 42): the first measure of "small changes in training, different portfolio". Filled in once seeds 0 and 5 are trained (stage 4 of Step 3).

In [7]:
seed_rows = []
for (model, universe) in sorted({(key[0], key[1]) for key in loaded}):
    seeds = sorted(key[2] for key in loaded if key[:2] == (model, universe))
    for i, a in enumerate(seeds):
        for b in seeds[i + 1:]:
            o = overlap(loaded[(model, universe, a)][2], loaded[(model, universe, b)][2])
            seed_rows.append({"model": model, "universe": universe, "seeds": f"{a} vs {b}", "overlap": o["jaccard"].mean(),
                              "turnover between seeds": o["turnover"].mean()})
pd.DataFrame(seed_rows) if seed_rows else print("Only one seed trained so far.")

Only one seed trained so far.


## 6. Saved summary

`results/analysis/portfolio_summary.csv`: the metrics of every portfolio and benchmark above, and `portfolio_margins.csv`.

In [8]:
pd.DataFrame(summary_rows).to_csv(RESULTS_ROOT / "analysis" / "portfolio_summary.csv", index=False)
margins.to_csv(RESULTS_ROOT / "analysis" / "portfolio_margins.csv", index=False)
print("saved")

saved


## Observations

OBSERVATIONS_PLACEHOLDER